In [ ]:
import torch
from torch import nn

device = 'cuda' if torch.cuda.is_available() else 'cpu'
world_size = 8
tokens = 5120
hiddens = 2048
total_vocab = 200021
tokens_per_gpu = tokens//world_size

embedding = nn.Embedding(total_vocab, hiddens, device=device)
sample_input = torch.randint(low=0, high=total_vocab, size=(tokens_per_gpu,), device=device).tolist()
input_tensor = torch.tensor(sample_input, dtype=torch.int32, device=device)

embedded_tensor = embedding(input_tensor)

input_q = embedded_tensor
input_k = embedded_tensor
input_v = embedded_tensor

torch.Size([8, 1280, 2048])

In [ ]:

def identify_nodes_for_qkv(world_size:int):
    assert world_size % 2 == 0
    min_partitons_per_node = ((world_size * (world_size+1))//2)//world_size
    work_per_node = {}
    exe_order_per_rank_v = [[] for _ in range(world_size)]
    exe_order_per_rank_h = [[] for _ in range(world_size)]
    for i in range(world_size):
        work_per_node[i] = min_partitons_per_node + (1 if i < world_size//2 else 0)
    for i in range(world_size):
        q_node_idx = i
        while work_per_node[i] > 0 and q_node_idx < world_size:
            exe_order_per_rank_v[i].append((q_node_idx,i))
            exe_order_per_rank_h[q_node_idx].append((q_node_idx, i))
            work_per_node[i] -= 1
            q_node_idx += 1
        dest_rank = world_size-1-i
        while q_node_idx < world_size:
            exe_order_per_rank_v[dest_rank].append((q_node_idx,i))
            q_node_idx += 1
    return exe_order_per_rank_v, exe_order_per_rank_h
    print(exe_order_per_rank_v)
    print(exe_order_per_rank_h)

        
identify_nodes_for_qkv(8)

[[(0, 0), (1, 0), (2, 0), (3, 0), (4, 0)], [(1, 1), (2, 1), (3, 1), (4, 1), (5, 1)], [(2, 2), (3, 2), (4, 2), (5, 2), (6, 2)], [(3, 3), (4, 3), (5, 3), (6, 3), (7, 3)], [(4, 4), (5, 4), (6, 4), (7, 4)], [(7, 2), (5, 5), (6, 5), (7, 5)], [(6, 1), (7, 1), (6, 6), (7, 6)], [(5, 0), (6, 0), (7, 0), (7, 7)]]
[[(0, 0)], [(1, 0), (1, 1)], [(2, 0), (2, 1), (2, 2)], [(3, 0), (3, 1), (3, 2), (3, 3)], [(4, 0), (4, 1), (4, 2), (4, 3), (4, 4)], [(5, 1), (5, 2), (5, 3), (5, 4), (5, 5)], [(6, 2), (6, 3), (6, 4), (6, 5), (6, 6)], [(7, 3), (7, 4), (7, 5), (7, 6), (7, 7)]]


In [ ]:
import torch
from torch import nn

# def _attention_forward_inner_non_mask(acc, l_i, m_i, q, desc_k, desc_v, 
#                              offset_y, dtype, start_m, qk_scale,
#                              block_m, hidden_dim, block_n,
#                              offs_m, offs_n, n_ctx, wrap_specialize):
#     lo, hi = 0, (start_m+1)*block_m
#     offsetk_y = offset_y + lo
#     offsetv_y = offset_y + lo
#     for start_n in torch.arange(lo, hi, block_n):
#         print(f"start_n : {start_n} : offset of k [{offsetk_y},0]")
#         print("no mask used")
#         print(f"Offset of v [0, {offsetv_y}]")
#         offsetk_y += block_n
#         offsetv_y += block_n

def _attention_forward_inner_mask(acc, l_i, m_i, q, desc_k, desc_v, 
                             offset_y, dtype, start_m, qk_scale,
                             block_m, hidden_dim, block_n, stage,
                             offs_m, offs_n, n_ctx, wrap_specialize):
    print(f"Stage : {stage}")
    if stage == 1:
        lo, hi = 0, start_m*block_m
    elif stage == 2:
        lo, hi = start_m*block_m, (start_m+1)*block_m
    else:
        print("Non-causal case")
        return
    offsetk_y = offset_y + lo
    offsetv_y = offset_y + lo
    for start_n in torch.arange(lo, hi, block_n):
        print(f"start_n : {start_n} : offset of k [{offsetk_y},0]")
        if stage == 2:
            mask = offs_m[:, None] >= (start_n + offs_n[None, :])
            print(f"mask is used {mask}")
        else:
            print("no mask used")
        print(f"Offset of v [0, {offsetv_y}]")
        offsetk_y += block_n
        offsetv_y += block_n
    


def _attention_forward(sm_scale, max_tensor, num_heads, n_ctx, desc_q, desc_k, desc_v, desc_o, hidden_dim, block_m, block_n, mask_region, wrap_specialize):
    dtype = torch.bfloat16
    assert block_n <= hidden_dim
    for start_m in range(20):
        for off_h in range(1):
            print(f"start_m : {start_m}, off_h : {off_h}")
            # y_dim = num_heads*n_ctx
            offset_y = off_h*n_ctx
            print(f"offset_y : {offset_y}")
            qo_offset_y = offset_y + start_m*block_m
            print(f"qo_offset_y : {qo_offset_y}")
            offs_m = start_m*block_m + torch.arange(0, block_m)
            offs_n = torch.arange(0, block_n)
            print(f"offs_m : {offs_m}")
            print(f"offs_n : {offs_n}")
            # initialize pointer to m and l
            m_i = torch.zeros([block_m], dtype=torch.float32) - float("inf")
            l_i = torch.zeros([block_m], dtype=torch.float32) + 1.0
            acc = torch.zeros([block_m, hidden_dim], dtype=torch.float32)
            # load scales
            qk_scale = sm_scale
            qk_scale *= 1.44269504 #1/log(2)
            # q = q.load([qo_offset_y,0])
            print(f"q load : {[qo_offset_y, 0]}")
            # q = torch.randn([qo_offset_y, 0], dtype=dtype, device=device, requires_grad=True)
            # if mask_region:
            _attention_forward_inner_mask(acc, l_i, m_i, q, desc_k, desc_v, offset_y, dtype, start_m, qk_scale, block_m, hidden_dim, block_n, 1, offs_m, offs_n, n_ctx, wrap_specialize)
            _attention_forward_inner_mask(acc, l_i, m_i, q, desc_k, desc_v, offset_y, dtype, start_m, qk_scale, block_m, hidden_dim, block_n, 2, offs_m, offs_n, n_ctx, wrap_specialize)
            # else:
            #     _attention_forward_inner_non_mask(acc, l_i, m_i, q, desc_k, desc_v, offset_y, dtype, start_m, qk_scale, block_m, hidden_dim, block_n, offs_m, offs_n, n_ctx, wrap_specialize)
             
        #     if off_h == 1:
        #         break
        # if start_m == 1:
        #     print("------------------------------")
        #     return

# Assumption that it is used only for causal case
def attention(q, k, v, num_heads, sm_scale, rank, wrap_specialize=True):
    for i in range(rank+1):
        
        HEAD_DIM_Q, HEAD_DIM_K, HEAD_DIM_V = q.shape[-1], k.shape[-1], v.shape[-1]
        q_with_head = q.unsqueeze(0).expand(num_heads, -1, -1)
        k_with_head = k.unsqueeze(0).expand(num_heads, -1, -1)
        v_with_head = v.unsqueeze(0).expand(num_heads, -1, -1)
        o_with_head = torch.empty_like(q_with_head)
        print(q_with_head.shape, k_with_head.shape, v_with_head.shape)
        BLOCK_M = 64
        BLOCK_N = 32
        M = torch.empty(q_with_head.shape[0], q_with_head.shape[1], device=device, dtype=torch.float32)
        print(f"Max tokens : {M.shape}")
        grid = (q_with_head.shape[1]//BLOCK_M, num_heads, 1)
        print(f"Grid : {grid}")
        # Attention forward
        if i == rank:
            # mask region
            _attention_forward(sm_scale, M, num_heads, q_with_head.shape[1],
                            q_with_head, k_with_head, v_with_head, o_with_head,
                            q_with_head.shape[-1], BLOCK_M, BLOCK_N, True, wrap_specialize)
        else:
            # non-mask region
            _attention_forward(sm_scale, M, num_heads, q_with_head.shape[1],
                            q_with_head, k_with_head, v_with_head, o_with_head,
                            q_with_head.shape[-1], BLOCK_M, BLOCK_N, False, wrap_specialize)

device = 'cuda' if torch.cuda.is_available() else 'cpu'
total_vocab = 200021
tokens_per_gpu = 1280
hiddens = 2048
n_heads = 1
q = torch.randint(low=0, high=total_vocab, size=(tokens_per_gpu,), device=device)
k = torch.randint(low=0, high=total_vocab, size=(tokens_per_gpu,), device=device)
v = torch.randint(low=0, high=total_vocab, size=(tokens_per_gpu,), device=device)
embedding = nn.Embedding(total_vocab, hiddens)
q_embed = embedding(q)
k_embed = embedding(k)
v_embed = embedding(v)
sm_scale = 0.5
rank = 0

attention(q_embed, k_embed, v_embed, n_heads, sm_scale, rank)



torch.Size([1, 1280, 2048]) torch.Size([1, 1280, 2048]) torch.Size([1, 1280, 2048])
Max tokens : torch.Size([1, 1280])
Grid : (20, 1, 1)
start_m : 0, off_h : 0
offset_y : 0
qo_offset_y : 0
offs_m : tensor([ 0,  1,  2,  3,  4,  5,  6,  7,  8,  9, 10, 11, 12, 13, 14, 15, 16, 17,
        18, 19, 20, 21, 22, 23, 24, 25, 26, 27, 28, 29, 30, 31, 32, 33, 34, 35,
        36, 37, 38, 39, 40, 41, 42, 43, 44, 45, 46, 47, 48, 49, 50, 51, 52, 53,
        54, 55, 56, 57, 58, 59, 60, 61, 62, 63])
offs_n : tensor([ 0,  1,  2,  3,  4,  5,  6,  7,  8,  9, 10, 11, 12, 13, 14, 15, 16, 17,
        18, 19, 20, 21, 22, 23, 24, 25, 26, 27, 28, 29, 30, 31])
q load : [0, 0]
Stage : 1
Stage : 2
start_n : 0 : offset of k [0,0]
mask is used tensor([[ True, False, False,  ..., False, False, False],
        [ True,  True, False,  ..., False, False, False],
        [ True,  True,  True,  ..., False, False, False],
        ...,
        [ True,  True,  True,  ...,  True,  True,  True],
        [ True,  True,  True,  ..

In [18]:
import torch
from torch import nn
from transformers import AutoTokenizer

device = 'cuda' if torch.cuda.is_available() else 'cpu'
tokens = 5120
hiddens = 2048
total_vocab = 200021
sample_input = torch.randint(low=0, high=total_vocab, size=(tokens,), device=device).tolist()
# print(sample_input)
embedding = nn.Embedding(total_vocab, hiddens)
input_tensor = torch.tensor(sample_input, dtype=torch.int32)
embedded_tensor = embedding(input_tensor)
print(embedded_tensor.shape)

torch.Size([5120, 2048])


In [5]:
device = 'cuda' if torch.cuda.is_available() else 'cpu'
tokens = 100
hiddens = 5 #2048
# input = torch.tensor([tokens], dtype=torch.long, device=device)
tokenizer_model_id = "openai/gpt-oss-20b"
bos_token = "<~!Start-of-Sentence!~>"
eos_token = "<~!End-of-Sentence!~>"
tokenizer = AutoTokenizer.from_pretrained(tokenizer_model_id,
                                          bos_token = bos_token,
                                          eos_token = eos_token)

In [6]:
input = "My name is debashis das"
encoded_tensor = torch.tensor(tokenizer.encode(input))
print(f"Input : {input}\nEncoded : {encoded_tensor}")
total_tokens = len(tokenizer)
print(f"Total tokens : {total_tokens}")

Input : My name is debashis das
Encoded : tensor([5444, 1308,  382, 4315, 1229,  276, 2331])
Total tokens : 200021


In [7]:
embeddings = nn.Embedding(total_tokens, hiddens)
embedded_input = embeddings(encoded_tensor)
print(f"input : {encoded_tensor.shape}\nembedding : {embedded_input.shape}")

input : torch.Size([7])
embedding : torch.Size([7, 5])


TODO Dataset conversion to tokens

In [11]:
tokens = 8
tokens_per_gpu = 2
world_size = 4
sample_input = torch.randn([tokens], device=device).tolist()
print(len(sample_input))
sample_per_gpu = []
output = [{idx:sample_input[i:i+tokens_per_gpu]} for idx,i in enumerate(range(0, tokens, tokens_per_gpu))]
print(output[0][0])

8
[0.9738694429397583, 1.2170926332473755]
